In [70]:
import os
import yaml
from pathlib import Path

config_path_obj = Path("config.yaml")

if not config_path_obj.exists():
    raise FileNotFoundError(f"config file not found : {config_path_obj}")

with open(config_path_obj, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file) or {}

In [71]:
import numpy as np
import pandas as pd

df = pd.read_csv("dataset/heart_failure_dataset.csv")
display(df.head())
display(df.describe())
display(df.info())

label_column = "HeartDisease"
categorical_columns = ["Sex","FastingBS","ExerciseAngina"]
ordinal_columns = ["ChestPainType","RestingECG","ST_Slope"]
nominal_columns = df.drop(columns=label_column).select_dtypes(include="str").columns.to_list()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364,0.553377
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570,0.497414
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000,0.000000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000,1.000000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000,1.000000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000,1.000000


<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    str    
 2   ChestPainType   918 non-null    str    
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    str    
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    str    
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    str    
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), str(5)
memory usage: 97.6 KB


None

In [72]:
from sklearn.model_selection import train_test_split

data_dict = config.get("data")
predictors = df.drop(columns=label_column)
label = df[label_column]
X_train, X_test, y_train, y_test = train_test_split(predictors,label,test_size=data_dict["test_size"],random_state=data_dict["split_seed"])

In [73]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import make_column_transformer

oneHotEncoder = OneHotEncoder(
    categories=categorical_columns,
    drop="first"
)

ordinal_dict = config.get("encoding")["ordinal_mappings"]
orders = [ordinal_dict["ChestPainType"], ordinal_dict["ST_Slope"], ordinal_dict["RestingECG"]]
print(orders)
ordinalEncoder = OrdinalEncoder(categories=orders)

preprocessor = make_column_transformer(
    (oneHotEncoder,categorical_columns),
    (ordinalEncoder,ordinal_columns),
    (StandardScaler(),nominal_columns),
    remainder="drop"
)


[['ASY', 'NAP', 'ATA', 'TA'], ['Up', 'Flat', 'Down'], ['Normal', 'ST', 'LVH']]


In [74]:
from sklearn.metrics import make_scorer, recall_score, fbeta_score
from sklearn.linear_model import LogisticRegressionCV
from sklearn.pipeline import make_pipeline

# Recall-first, but precision-aware (beta<2 makes precision matter more than pure recall scoring)
scorer = make_scorer(
    fbeta_score,
    beta=2,
    pos_label=1,
    average="binary"
)

# Fix preprocessing definitions
categorical_columns = ["Sex", "FastingBS", "ExerciseAngina"]
ordinal_columns = ["ChestPainType", "RestingECG", "ST_Slope"]
numeric_columns = X_train.drop(columns=categorical_columns + ordinal_columns).columns.to_list()

oneHotEncoder = OneHotEncoder(
    drop="first",
    handle_unknown="ignore"
)

ordinal_dict = config.get("encoding")["ordinal_mappings"]
orders = [
    ordinal_dict["ChestPainType"],
    ordinal_dict["RestingECG"],
    ordinal_dict["ST_Slope"]
]
ordinalEncoder = OrdinalEncoder(
    categories=orders,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

preprocessor = make_column_transformer(
    (oneHotEncoder, categorical_columns),
    (ordinalEncoder, ordinal_columns),
    (StandardScaler(), numeric_columns),
    remainder="drop"
)

logisticRegressionCV = LogisticRegressionCV(
    Cs=20,
    cv=5,
    l1_ratios=(0.0,),
    scoring=scorer,
    class_weight={0: 1.0, 1: 2},
    solver="lbfgs",
    max_iter=2000,
    random_state=data_dict["split_seed"],
    n_jobs=-1,
    use_legacy_attributes=False,
)

pipeline = make_pipeline(
    preprocessor,
    logisticRegressionCV
)

_ = pipeline.fit(X_train, y_train)

In [75]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

# Evaluate on test set
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

print("Evaluation metrics:")
for k, v in metrics.items():
    print(f"- {k}: {v:.4f}")

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4, zero_division=0))

Evaluation metrics:
- accuracy: 0.8641
- precision: 0.8942
- recall: 0.8692
- f1: 0.8815
- roc_auc: 0.9063

Confusion matrix:
[[66 11]
 [14 93]]

Classification report:
              precision    recall  f1-score   support

           0     0.8250    0.8571    0.8408        77
           1     0.8942    0.8692    0.8815       107

    accuracy                         0.8641       184
   macro avg     0.8596    0.8632    0.8611       184
weighted avg     0.8653    0.8641    0.8645       184



In [76]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, fbeta_score

# Recall-first scorer with precision awareness (same spirit as your current setup)
tune_scorer = make_scorer(fbeta_score, beta=1.5, pos_label=1, average="binary")

# Baseline params inspired by your current LogisticRegressionCV cell
base_lr = LogisticRegression(
    solver="lbfgs",
    max_iter=1500,
    random_state=data_dict["split_seed"],
    class_weight={0: 1.0, 1: 1.4},
)

lr_tune_pipeline = make_pipeline(preprocessor, base_lr)

lr_param_grid = {
    "logisticregression__C": [0.1, 0.3, 1.0, 3.0, 10.0],
    "logisticregression__class_weight": [
        {0: 1.0, 1: 1.2},
        {0: 1.0, 1: 1.4},
        {0: 1.0, 1: 1.6},
    ],
}

lr_grid = GridSearchCV(
    estimator=lr_tune_pipeline,
    param_grid=lr_param_grid,
    scoring=tune_scorer,
    cv=5,
    n_jobs=-1,
    refit=True,
)

lr_grid.fit(X_train, y_train)

best_lr_model = lr_grid.best_estimator_
print("Best Logistic params:", lr_grid.best_params_)
print(f"Best Logistic CV score (F-beta, beta=1.5): {lr_grid.best_score_:.4f}")

Best Logistic params: {'logisticregression__C': 0.3, 'logisticregression__class_weight': {0: 1.0, 1: 1.6}}
Best Logistic CV score (F-beta, beta=1.5): 0.8937


In [77]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.metrics import make_scorer, fbeta_score

rf_tune_scorer = make_scorer(fbeta_score, beta=1.5, pos_label=1, average="binary")

rf_pipeline = make_pipeline(
    preprocessor,
    RandomForestClassifier(random_state=data_dict["split_seed"], n_jobs=-1)
)

rf_param_grid = {
    "randomforestclassifier__n_estimators": [200, 400],
    "randomforestclassifier__max_depth": [None, 8, 12],
    "randomforestclassifier__min_samples_split": [2, 8],
    "randomforestclassifier__min_samples_leaf": [1, 3],
    "randomforestclassifier__class_weight": ["balanced", {0: 1.0, 1: 1.2}, {0: 1.0, 1: 1.5}],
}

rf_grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring=rf_tune_scorer,
    cv=5,
    n_jobs=-1,
    refit=True,
)

rf_grid.fit(X_train, y_train)

best_rf_model = rf_grid.best_estimator_
print("Best RandomForest params:", rf_grid.best_params_)
print(f"Best RandomForest CV score (F-beta, beta=1.5): {rf_grid.best_score_:.4f}")

Best RandomForest params: {'randomforestclassifier__class_weight': {0: 1.0, 1: 1.5}, 'randomforestclassifier__max_depth': 12, 'randomforestclassifier__min_samples_leaf': 3, 'randomforestclassifier__min_samples_split': 2, 'randomforestclassifier__n_estimators': 400}
Best RandomForest CV score (F-beta, beta=1.5): 0.9008


In [78]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

def evaluate_model(model, X_test, y_test, name: str):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    return {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision_label_1": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "recall_label_1": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "f1_label_1": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
        "classification_report": classification_report(y_test, y_pred, digits=4, zero_division=0),
    }

results = [
    evaluate_model(best_lr_model, X_test, y_test, "Logistic (tuned)"),
    evaluate_model(best_rf_model, X_test, y_test, "RandomForest (tuned)"),
]

report_df = pd.DataFrame(results).drop(columns=["classification_report"])
display(report_df)

for row in results:
    print(f"\n{row['model']} confusion matrix:")
    print(row["confusion_matrix"])
    print(f"\n{row['model']} classification report:")
    print(row["classification_report"])

,model,accuracy,precision_label_1,recall_label_1,f1_label_1,roc_auc,confusion_matrix
0,Logistic (tuned),0.858696,0.900990,0.850467,0.875000,0.906057,"[[67, 10], [16, 91]]"
1,RandomForest (tuned),0.896739,0.907407,0.915888,0.911628,0.938949,"[[67, 10], [9, 98]]"



Logistic (tuned) confusion matrix:
[[67, 10], [16, 91]]

Logistic (tuned) classification report:
              precision    recall  f1-score   support

           0     0.8072    0.8701    0.8375        77
           1     0.9010    0.8505    0.8750       107

    accuracy                         0.8587       184
   macro avg     0.8541    0.8603    0.8562       184
weighted avg     0.8618    0.8587    0.8593       184


RandomForest (tuned) confusion matrix:
[[67, 10], [9, 98]]

RandomForest (tuned) classification report:
              precision    recall  f1-score   support

           0     0.8816    0.8701    0.8758        77
           1     0.9074    0.9159    0.9116       107

    accuracy                         0.8967       184
   macro avg     0.8945    0.8930    0.8937       184
weighted avg     0.8966    0.8967    0.8966       184



In [84]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import re
import mlflow
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient

tracking_dir = Path("mlruns").resolve()
mlflow.set_tracking_uri(tracking_dir.as_uri())

client = MlflowClient()


def _slug(text: str) -> str:
    text = str(text).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_") or "na"


def get_or_restore_experiment(name: str):
    experiments = client.search_experiments(view_type=ViewType.ALL)

    for exp in experiments:
        if exp.name == name:
            if exp.lifecycle_stage == "deleted":
                print(f"Restoring deleted experiment: {name}")
                client.restore_experiment(exp.experiment_id)
            return exp.experiment_id

    print(f"Creating experiment: {name}")
    return client.create_experiment(name)


label_slug = _slug(config.get("data", {}).get("label_column", "heart_disease"))
run_slug = _slug("basemodels_tuned")
EXPERIMENT_NAME = f"iomt_ml.{label_slug}.{run_slug}.tuned"
session_uid = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid4().hex[:8]}"

experiment_id = get_or_restore_experiment(EXPERIMENT_NAME)
mlflow.set_experiment(experiment_id=experiment_id)

# ---------------------------------------------------
# Safety checks
# ---------------------------------------------------
if "results" not in globals():
    raise RuntimeError("Run the evaluation cell first so 'results' exists.")

model_lookup = {
    "Logistic (tuned)": best_lr_model,
    "RandomForest (tuned)": best_rf_model,
}

# ---------------------------------------------------
# Logging loop
# ---------------------------------------------------
for row in results:

    model_name = row["model"]

    if model_name not in model_lookup:
        print(f"Skipping unknown model: {model_name}")
        continue

    model_slug = _slug(model_name)
    run_name = f"{run_slug}.{model_slug}.{session_uid}"
    model_artifact_name = f"{model_slug}_{session_uid}"

    with mlflow.start_run(run_name=run_name):

        mlflow.set_tag("notebook", "basemodel_trainer")
        mlflow.set_tag("experiment_name", EXPERIMENT_NAME)
        mlflow.set_tag("session_uid", session_uid)
        mlflow.set_tag("model_family", model_name)
        mlflow.set_tag("model_artifact_name", model_artifact_name)

        # ---------- Metrics ----------
        metrics = {
            "accuracy": row["accuracy"],
            "precision_label_1": row["precision_label_1"],
            "recall_label_1": row["recall_label_1"],
            "f1_label_1": row["f1_label_1"],
            "roc_auc": row["roc_auc"],
        }

        mlflow.log_metrics({k: float(v) for k, v in metrics.items()})

        # ---------- Confusion matrix ----------
        cm = row.get("confusion_matrix")
        if (
            isinstance(cm, list)
            and len(cm) == 2
            and len(cm[0]) == 2
            and len(cm[1]) == 2
        ):
            mlflow.log_metrics(
                {
                    "tn": float(cm[0][0]),
                    "fp": float(cm[0][1]),
                    "fn": float(cm[1][0]),
                    "tp": float(cm[1][1]),
                }
            )

        # ---------- Classification report ----------
        report_path = Path(f"classification_report_{model_slug}_{session_uid}.txt")
        report_path.write_text(row["classification_report"], encoding="utf-8")

        mlflow.log_artifact(str(report_path))
        report_path.unlink(missing_ok=True)

        # ---------- Best params ----------
        if model_name == "Logistic (tuned)" and "lr_grid" in globals():
            mlflow.log_params(
                {f"best_{k}": str(v) for k, v in lr_grid.best_params_.items()}
            )

        if model_name == "RandomForest (tuned)" and "rf_grid" in globals():
            mlflow.log_params(
                {f"best_{k}": str(v) for k, v in rf_grid.best_params_.items()}
            )

        # ---------- Model ----------
        mlflow.sklearn.log_model(model_lookup[model_name], name=model_artifact_name)

        print(f"Logged to MLflow: run={run_name}, model={model_artifact_name}")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"MLflow experiment: {EXPERIMENT_NAME}")
print(f"MLflow session UID: {session_uid}")

Creating experiment: iomt_ml.heartdisease.basemodels_tuned.tuned


2026/03/31 06:08:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged to MLflow: run=basemodels_tuned.logistic_tuned.20260330T220816Z_a71ed44a, model=logistic_tuned_20260330T220816Z_a71ed44a


2026/03/31 06:08:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged to MLflow: run=basemodels_tuned.randomforest_tuned.20260330T220816Z_a71ed44a, model=randomforest_tuned_20260330T220816Z_a71ed44a
MLflow tracking URI: file:///C:/Users/BIZZ/Documents/Vscode/python/IoMT%20ML/mlruns
MLflow experiment: iomt_ml.heartdisease.basemodels_tuned.tuned
MLflow session UID: 20260330T220816Z_a71ed44a
